## Task-5: Fine-tune a transformer model (BERT/DistilBERT) to perform: POS Tagging and Chunking 

# Objective:
Fine-tune a transformer model (DistilBERT) for token classification to perform
Part-of-Speech tagging and Chunking using the CoNLL-2003 dataset.

## Step-1: Install Required Libraries

In [1]:
!pip install transformers datasets seqeval torch

     ---------------------------------------- 0.0/43.6 kB ? eta -:--:--
     --------- ------------------------------ 10.2/43.6 kB ? eta -:--:--
     ----------------- -------------------- 20.5/43.6 kB 217.9 kB/s eta 0:00:01
     ----------------------------------- -- 41.0/43.6 kB 326.8 kB/s eta 0:00:01
     -------------------------------------- 43.6/43.6 kB 236.6 kB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16282 sha256=2edc117740b544e35fd1d771fb8e68d8cd05a200f7b90a97012d63fa58ff2a61
  Stored in directory: c:\u


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


##  Step-2: Import Libraries

In [2]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer

from seqeval.metrics import classification_report

c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Task-1: Dataset Selection

In [4]:
!pip install datasets==2.16.1

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/507.1 kB ? eta -:--:--
   ----- ---------------------------------- 71.7/507.1 kB 1.3 MB/s eta 0:00:01
   ------------ --------------------------- 153.6/507.1 kB 1.3 MB/s eta 0:00:01
   -------------------- ------------------- 256.0/507.1 kB 1.6 MB/s eta 0:00:01
   -------------------------- ------------- 337.9/507.1 kB 1.7 MB/s eta 0:00:01
   ------------------------------------- -- 471.0/507.1 kB 1.8 MB/s eta 0:00:01
   ---------------------------------------  501.8/507.1 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 507.1/507.1 kB 1.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/115.3 kB ? eta -:--:--
   ---------------------------------------  112.6/115.3 kB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 115.3/115.3 kB 1.7 MB/s et


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

dataset

c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\datasets\load.py:1429: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
Generating test split: 100%|██████████| 3453/3453 [00:00<00:00, 9270.64 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

## Label Categories

In [2]:
pos_labels = dataset["train"].features["pos_tags"].feature.names
chunk_labels = dataset["train"].features["chunk_tags"].feature.names

pos_labels

['"',
 "''",
 '#',
 '$',
 '(',
 ')',
 ',',
 '.',
 ':',
 '``',
 'CC',
 'CD',
 'DT',
 'EX',
 'FW',
 'IN',
 'JJ',
 'JJR',
 'JJS',
 'LS',
 'MD',
 'NN',
 'NNP',
 'NNPS',
 'NNS',
 'NN|SYM',
 'PDT',
 'POS',
 'PRP',
 'PRP$',
 'RB',
 'RBR',
 'RBS',
 'RP',
 'SYM',
 'TO',
 'UH',
 'VB',
 'VBD',
 'VBG',
 'VBN',
 'VBP',
 'VBZ',
 'WDT',
 'WP',
 'WP$',
 'WRB']

In [3]:
pos_labels = dataset["train"].features["pos_tags"].feature.names
chunk_labels = dataset["train"].features["chunk_tags"].feature.names

pos_labels

['"',
 "''",
 '#',
 '$',
 '(',
 ')',
 ',',
 '.',
 ':',
 '``',
 'CC',
 'CD',
 'DT',
 'EX',
 'FW',
 'IN',
 'JJ',
 'JJR',
 'JJS',
 'LS',
 'MD',
 'NN',
 'NNP',
 'NNPS',
 'NNS',
 'NN|SYM',
 'PDT',
 'POS',
 'PRP',
 'PRP$',
 'RB',
 'RBR',
 'RBS',
 'RP',
 'SYM',
 'TO',
 'UH',
 'VB',
 'VBD',
 'VBG',
 'VBN',
 'VBP',
 'VBZ',
 'WDT',
 'WP',
 'WP$',
 'WRB']

## Task-2: Data Preprocessing (Tokenization + Alignment)

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sweth\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


# POS Tokenization

In [15]:
def tokenize_pos(examples):
    tokenized_inputs = tokenizer(
    examples["tokens"],
    truncation=True,
    padding="max_length",
    max_length=128,
    is_split_into_words=True
)
    

    labels = []

    for i, label in enumerate(examples["pos_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply Preprocessing

In [ ]:
pos_tokenized = dataset.map(tokenize_pos, batched=True)

Map: 100%|██████████| 3453/3453 [00:00<00:00, 5717.27 examples/s]


## Task-3: Model Setup

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(pos_labels)
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3169.22it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Task-4: Training

In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=0.2,
    weight_decay=0.01
)

# Trainer

In [23]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=pos_tokenized["train"],
    eval_dataset=pos_tokenized["validation"]
)

In [24]:
trainer.train()

c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]


TrainOutput(global_step=352, training_loss=0.44409019296819513, metrics={'train_runtime': 2033.5258, 'train_samples_per_second': 1.381, 'train_steps_per_second': 0.173, 'total_flos': 92054622240768.0, 'train_loss': 0.44409019296819513, 'epoch': 0.20045558086560364})

## Task-5: Evaluation

In [25]:
predictions = trainer.predict(pos_tokenized["validation"])

c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [26]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=2)
labels = predictions.label_ids

In [28]:
true_labels = []
true_predictions = []

for pred, label in zip(preds, labels):
    true_labels.append([pos_labels[l] for l in label if l != -100])
    true_predictions.append([pos_labels[p] for (p, l) in zip(pred, label) if l != -100])

In [29]:
from seqeval.metrics import classification_report

print(classification_report(true_labels, true_predictions))

c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\sweth\AppData\Local\Programs\Python\Python311\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: . seems not 

              precision    recall  f1-score   support

           '       0.00      0.00      0.00        11
           B       0.79      0.81      0.80      1907
          BD       0.92      0.95      0.93      2224
          BG       0.90      0.78      0.84       699
          BN       0.86      0.79      0.82       928
          BP       0.88      0.77      0.82       365
          BR       0.00      0.00      0.00        53
          BS       0.00      0.00      0.00        18
          BZ       0.90      0.90      0.90       509
           C       0.99      1.00      1.00       932
           D       0.89      0.97      0.93      3229
          DT       0.85      0.83      0.84       162
           H       0.00      0.00      0.00         5
           J       0.73      0.74      0.74      2764
          JR       1.00      0.03      0.06       105
          JS       0.00      0.00      0.00        78
           N       0.84      0.85      0.84      8147
          NP       0.78    

## Step-6: Inference

In [30]:
sentence = "John works at Google in California"
words = sentence.split()

inputs = tokenizer(
    words,
    is_split_into_words=True,
    return_tensors="pt",
    truncation=True,
    padding=True
)

outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2)

predicted_labels = [
    pos_labels[p.item()]
    for p in predictions[0]
]

for word, label in zip(words, predicted_labels[1:len(words)+1]):
    print(word, "->", label)

John -> NNP
works -> VBZ
at -> IN
Google -> NNP
in -> IN
California -> NNP


## Task-7: Comparision

 ## Comparison: POS Tagging vs Chunking

 # POS Tagging (Grammar-Level Tagging — Easy)

Part-of-Speech (POS) tagging assigns grammatical labels to each individual word in a sentence. These labels represent the syntactic role of words such as noun, verb, adjective, adverb, pronoun, etc. POS tagging works at the word level and does not consider phrase grouping. Since each token receives only one label, POS tagging is considered easier compared to chunking.

Example:

Sentence:

John works at Google in California

POS Tagging Output:

John → NNP  
works → VBZ  
at → IN  
Google → NNP  
in → IN  
California → NNP

POS tagging helps in:

Grammar understanding
Named entity recognition preprocessing
Information extraction
Language modeling


# Chunking (Phrase-Level Grouping — Medium)

Chunking groups words into meaningful phrases such as noun phrases (NP), verb phrases (VP), and prepositional phrases (PP). Unlike POS tagging, chunking works at the phrase level instead of individual words. Chunking depends on POS tags and sentence structure, making it more complex than POS tagging.

Example:

Sentence:

John works at Google in California

Chunking Output:

John → B-NP  
works → B-VP  
at → B-PP  
Google → B-NP  
in → B-PP  
California → B-NP

Chunking helps in:

Phrase detection
Syntax understanding
Information extraction

Summary:

POS Tagging assigns grammatical labels to each word such as noun, verb, adjective, etc. It focuses on word-level classification and is considered easier. 

Chunking groups words into meaningful phrases like noun phrase (NP) or verb phrase (VP). 

Chunking is more complex because it depends on POS tags and sentence structure. 

POS tagging provides grammatical understanding while chunking provides phrase-level understanding.

## Task-8: Report/ Blog

## Introduction

In this task, a transformer-based model (BERT/DistilBERT) was fine-tuned for token classification to perform Part-of-Speech (POS) tagging and Chunking. Token classification assigns a label to each token in a sentence. POS tagging focuses on grammatical roles, while chunking groups tokens into meaningful phrases. The model was trained using the CoNLL dataset and evaluated using sequence-based metrics.

Difference Between POS Tagging and Chunking:

1. POS Tagging
Assigns grammatical labels to each word
Works at word level
Identifies nouns, verbs, adjectives, etc.
Easier task compared to chunking
Example:

Sentence:

John works at Google

POS Output:

John → NNP  
works → VBZ  
at → IN  
Google → NNP

2. Chunking
Groups words into phrases
Works at phrase level
Uses POS tags internally
More complex than POS tagging
Example:

Sentence:

John works at Google

Chunk Output:

John → B-NP  
works → B-VP  
at Google → B-PP B-NP

Observations:

Transformer models perform well for sequence labeling tasks
BERT captures contextual meaning of words effectively
Label alignment is important after tokenization
Subword tokenization affects label mapping
Training time is higher on CPU environment
Evaluation metrics show good micro F1 score
Model generalizes well on validation data
Token classification pipeline works efficiently


Challenges Faced:

Handling subword tokenization using word_ids()
Aligning labels with tokenized inputs
Managing special tokens like [CLS] and [SEP]
Assigning -100 to ignored tokens
Long training time without GPU
Formatting labels for seqeval evaluation
Converting predictions to label names
Handling padding during training


Insights:

Label alignment is critical for token classification
Incorrect alignment reduces model performance
BERT improves contextual understanding
Trainer API simplifies training process
seqeval is useful for sequence evaluation
POS tagging is easier than chunking

# Conclusion:

In this task, a transformer model was successfully fine-tuned for POS tagging and chunking using token classification. The model achieved good evaluation scores and correctly predicted labels for custom input sentences. The experiment demonstrates that BERT-based models are effective for sequence labeling tasks. POS tagging provides grammatical information, while chunking provides phrase-level understanding. Overall, the implementation shows the power of transformers in NLP token classification tasks.